# EDA of match_features

What in `match_features.csv` says anything about the result, before kick-off?
Run `uv run collect_data` and `uv run clean_data` first; this notebook only reads
`match_features.csv` and `missing.csv` from `settings.clean.out_dir`.

0. Columns: the target, the features, and the columns that leak the result
1. The target, and the baseline any model has to beat
2. What is missing, and whether being missing says something
3. One feature at a time: drift across seasons, and its link to H/D/A
4. Features together: overlap, and what is left once Elo is known
5. Summary for feature_importance (issue #13)

Everything below looks only at the training seasons (`season_start < TEST_FROM`),
so the test seasons stay unseen until the model is scored on them.

In [ ]:
import re

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.preprocessing import StandardScaler

from money_printer_research.analysis.feature_importance import POST_MATCH, TEST_FROM, feature_group
from money_printer_research.config import settings
from money_printer_research.schema import FormationCols, TeamFatigueCols, TeamScheduleCols
from money_printer_research.schema.match.features import PLAYER_SLOT_FEATURES, slot_columns

pl.Config.set_tbl_cols(20)
pl.Config.set_tbl_rows(30)
plt.rcParams["figure.dpi"] = 110

TARGET = "full_time_result"
OUTCOMES = ["H", "D", "A"]

matches = pl.read_csv(
    settings.clean.out_dir / "match_features.csv", infer_schema_length=None, try_parse_dates=True
)
missing = pl.read_csv(settings.clean.out_dir / "missing.csv")
print(matches.shape, missing.shape)

## 0. Columns

Features are listed by where they come from, not by "every column but a few": a new
column then has to be added here on purpose. The goals and xG of the match itself
are known only after it, so they must never be features; the assert below fails if
one slips in.

In [ ]:
ELO = ["elo_home", "elo_away", "elo_diff"]
FATIGUE = [c for c in TeamFatigueCols.to_schema().columns if c != "game_id"]
SCHEDULE = [c for c in TeamScheduleCols.to_schema().columns if c != "game_id"]
FORMATION = [
    c
    for c, col in FormationCols.to_schema().columns.items()
    if col.dtype is not None and str(col.dtype) not in {"String", "Utf8"}
]
SLOTS = list(slot_columns())
FEATURES = [c for c in [*ELO, *FATIGUE, *SCHEDULE, *FORMATION, *SLOTS] if c in matches.columns]

LEAKS = re.compile(POST_MATCH.pattern + r"|^(home|away)_goals$|^tm_(home|away)_goals$|^is_result$")
leaking = [c for c in FEATURES if LEAKS.match(c)]
assert not leaking, leaking

absent = sorted({*ELO, *FATIGUE, *SCHEDULE, *FORMATION, *SLOTS} - set(matches.columns))
print("features:", len(FEATURES), "| expected but absent:", absent)
print("never features (post-match):", [c for c in matches.columns if LEAKS.match(c)])

pl.DataFrame({"column": FEATURES}).group_by(
    pl.col("column").map_elements(feature_group, return_dtype=pl.String).alias("group")
).len().sort("len", descending=True)

In [ ]:
train = matches.filter(pl.col("season_start") < TEST_FROM)
print("train:", train.height, "games,", train["season_start"].n_unique(), "seasons")
print("held out (not looked at):", matches.height - train.height, "games")

## 1. The target

Share of H/D/A per league and season. The 2020/21 seasons were played without crowds:
if home advantage is real, the home share should dip there.

In [ ]:
shares = (
    train.group_by("league", "season_start", TARGET)
    .len()
    .with_columns(
        (pl.col("len") / pl.col("len").sum().over("league", "season_start")).alias("share")
    )
    .pivot(on=TARGET, index=["league", "season_start"], values="share")
    .select("league", "season_start", *OUTCOMES)
    .sort("league", "season_start")
)
shares

In [ ]:
leagues = shares["league"].unique().sort().to_list()
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5), sharey=True)
for ax, outcome in zip(axes, OUTCOMES, strict=True):
    for league in leagues:
        s = shares.filter(pl.col("league") == league)
        ax.plot(s["season_start"], s[outcome], marker="o", label=league)
    ax.axvspan(2019.5, 2020.5, color="grey", alpha=0.15)
    ax.set_title(f"share of {outcome}")
    ax.set_xlabel("season start")
axes[0].legend(fontsize=7)
plt.tight_layout()

**Baseline.** Predict every game with the training shares of H/D/A and nothing else.
Its log loss is the bar every model has to clear; a feature that cannot beat it is noise.
Scored season by season, with shares from the seasons before it only.

In [ ]:
seasons = train["season_start"].unique().sort().to_list()
rows = []
for s in seasons[1:]:
    past = train.filter(pl.col("season_start") < s)
    now = train.filter(pl.col("season_start") == s)
    prior = past[TARGET].value_counts(normalize=True)
    labels = sorted(OUTCOMES)  # log_loss reads probability columns in sorted label order
    p = [prior.filter(pl.col(TARGET) == o)["proportion"].sum() for o in labels]
    rows.append(
        {
            "season_start": s,
            "games": now.height,
            "log_loss_prior": log_loss(now[TARGET], [p] * now.height, labels=labels),
        }
    )
baseline = pl.DataFrame(rows)
baseline

## 2. What is missing

`missing.csv` has one row per null cell of match_features and why it is null.
`plain_shape` is expected (a formation with no variant) and left out here.

In [ ]:
cells = missing.filter(pl.col("reason") != "plain_shape").join(
    matches.select("game_id", "league", "season_start", TARGET), on="game_id"
)
by_season = (
    cells.group_by("reason", "season_start")
    .len()
    .pivot(on="season_start", index="reason", values="len", sort_columns=True)
    .fill_null(0)
    .sort("reason")
)
by_season

In [ ]:
grid = by_season.drop("reason").to_numpy()
fig, ax = plt.subplots(figsize=(12, 0.5 * len(grid) + 1))
im = ax.imshow(np.log1p(grid), aspect="auto", cmap="viridis")
ax.set_yticks(range(len(grid)), by_season["reason"].to_list())
ax.set_xticks(range(grid.shape[1]), by_season.columns[1:], rotation=45)
ax.set_title("null cells per reason and season (log scale)")
fig.colorbar(im, ax=ax)
plt.tight_layout()

In [ ]:
cells.group_by("reason", "league").len().pivot(
    on="league", index="reason", values="len", sort_columns=True
).fill_null(0).sort("reason")

**Does being missing say something?** For each reason, compare H/D/A in the games that
have it with the games that do not. A clear gap means the null is itself a signal,
worth a flag column in the model rather than a value to fill.

In [ ]:
def result_shares(df: pl.DataFrame) -> dict[str, float]:
    counts = df[TARGET].value_counts(normalize=True)
    return {o: counts.filter(pl.col(TARGET) == o)["proportion"].sum() for o in OUTCOMES}


train_cells = cells.filter(pl.col("season_start") < TEST_FROM)
rows = []
for reason in train_cells["reason"].unique().sort():
    ids = train_cells.filter(pl.col("reason") == reason)["game_id"].unique()
    has = train.filter(pl.col("game_id").is_in(ids.implode()))
    rest = train.filter(~pl.col("game_id").is_in(ids.implode()))
    rows.append(
        {"reason": reason, "games": has.height}
        | {f"{o}_with": v for o, v in result_shares(has).items()}
        | {f"{o}_without": v for o, v in result_shares(rest).items()}
    )
pl.DataFrame(rows)

**Player slots.** `too_few_minutes` is the biggest reason: a starter with too few
previous minutes has no per-90 rates. How often is each slot usable?

In [ ]:
slot_nulls = (
    train.select([c for c in SLOTS if c in train.columns])
    .null_count()
    .unpivot(variable_name="column", value_name="nulls")
    .with_columns(
        pl.col("column").str.extract(r"_player\d+_(.+)$").alias("feature"),
        pl.col("column").str.extract(r"_player(\d+)_").cast(pl.Int32).alias("slot"),
        (pl.col("nulls") / train.height).alias("null_share"),
    )
)
slot_nulls.pivot(on="feature", index="slot", values="null_share", aggregate_function="mean").sort(
    "slot"
)

## 3. One feature at a time

Home-minus-away differences first: the result depends on the gap between the sides
more than on either side alone. The player slots are averaged over the starters of
each side (`h_xg90_mean`), since 11 correlated slots spread the signal thin.

Elo starts every team level in 2014/15, so that season is warm-up and left out of
anything that uses Elo.

In [ ]:
def side_mean(side: str, feature: str) -> pl.Expr:
    cols = [
        c for c in SLOTS if re.fullmatch(rf"{side}_player\d+_{feature}", c) and c in train.columns
    ]
    return pl.mean_horizontal(cols).alias(f"{side}_{feature}_mean")


def with_derived(df: pl.DataFrame) -> pl.DataFrame:
    df = df.with_columns(side_mean(side, f) for side in ("h", "a") for f in PLAYER_SLOT_FEATURES)
    return df.with_columns(
        (pl.col("h_rest_days_all") - pl.col("a_rest_days_all")).alias("rest_diff"),
        (pl.col("h_games_14d_all") - pl.col("a_games_14d_all")).alias("games_14d_diff"),
        (pl.col("h_min_7d") - pl.col("a_min_7d")).alias("min_7d_diff"),
        *[
            (pl.col(f"h_{f}_mean") - pl.col(f"a_{f}_mean")).alias(f"{f}_diff")
            for f in PLAYER_SLOT_FEATURES
        ],
    )


eda = with_derived(train).filter(pl.col("season_start") > train["season_start"].min())
KEY = [
    "elo_diff",
    "fatigue_diff",
    "rest_diff",
    "games_14d_diff",
    "min_7d_diff",
    *[f"{f}_diff" for f in PLAYER_SLOT_FEATURES],
]
eda.select(KEY).describe()

**Drift.** The median and spread of each feature per season. A feature whose scale moves
from season to season (more European games, a change in how Understat records minutes)
will not mean the same thing in the test seasons.

In [ ]:
eda.group_by("season_start").agg(*[pl.col(c).median().alias(c) for c in KEY]).sort("season_start")

In [ ]:
fig, axes = plt.subplots(2, (len(KEY) + 1) // 2, figsize=(16, 6))
for ax, col in zip(axes.flat, KEY, strict=False):
    per_season = [
        eda.filter(pl.col("season_start") == s)[col].drop_nulls().to_numpy()
        for s in eda["season_start"].unique().sort()
    ]
    ax.boxplot(per_season, showfliers=False)
    ax.set_title(col, fontsize=9)
    ax.tick_params(labelsize=6)
for ax in axes.flat[len(KEY) :]:
    ax.set_visible(False)
plt.tight_layout()

**Link to the result.** Cut each feature into deciles and plot P(H), P(D), P(A) per decile.
A flat line says nothing; a slope says the feature moves the result. `elo_diff` is the
reference: every other feature is judged against how steep it is.

In [ ]:
def by_decile(df: pl.DataFrame, col: str, q: int = 10) -> pl.DataFrame:
    return (
        df.filter(pl.col(col).is_not_null())
        .with_columns(pl.col(col).qcut(q, allow_duplicates=True).alias("bin"))
        .group_by("bin")
        .agg(
            pl.col(col).mean().alias("x"),
            pl.len().alias("games"),
            *[(pl.col(TARGET) == o).mean().alias(o) for o in OUTCOMES],
        )
        .sort("x")
    )


fig, axes = plt.subplots(2, (len(KEY) + 1) // 2, figsize=(16, 6), sharey=True)
for ax, col in zip(axes.flat, KEY, strict=False):
    d = by_decile(eda, col)
    for o in OUTCOMES:
        ax.plot(d["x"], d[o], marker=".", label=o)
    ax.set_title(col, fontsize=9)
for ax in axes.flat[len(KEY) :]:
    ax.set_visible(False)
axes.flat[0].legend()
plt.tight_layout()

**Formations.** H/D/A by the home side's shape, for shapes with enough games.

In [ ]:
(
    eda.filter(pl.col("home_shape").is_not_null())
    .group_by("home_shape")
    .agg(pl.len().alias("games"), *[(pl.col(TARGET) == o).mean().alias(o) for o in OUTCOMES])
    .filter(pl.col("games") >= 100)
    .sort("games", descending=True)
)

## 4. Features together

**Overlap.** Correlations between the key features, on the games where all are present.
Two features that move together carry the same information; keep the cleaner one.

In [ ]:
complete = eda.select(KEY).drop_nulls()
corr = complete.corr()
print("games with every key feature:", complete.height, "of", eda.height)
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr.to_numpy(), cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(KEY)), KEY, rotation=60, ha="right", fontsize=8)
ax.set_yticks(range(len(KEY)), KEY, fontsize=8)
for i in range(len(KEY)):
    for j in range(len(KEY)):
        ax.text(j, i, f"{corr[i, j]:.2f}", ha="center", va="center", fontsize=6)
fig.colorbar(im, ax=ax)
plt.tight_layout()

**What is left once Elo is known?** Big clubs both rate high and play in Europe, so
fatigue or schedule can look useful only because they stand in for Elo. Within each
fifth of `elo_diff`, does the feature still change P(H)?

In [ ]:
def within_elo(df: pl.DataFrame, col: str) -> pl.DataFrame:
    return (
        df.filter(pl.col(col).is_not_null())
        .with_columns(
            pl.col("elo_diff").qcut(5, labels=[f"elo q{i}" for i in range(1, 6)]).alias("elo"),
            pl.col(col).qcut(3, labels=["low", "mid", "high"], allow_duplicates=True).alias(col),
        )
        .group_by("elo", col)
        .agg((pl.col(TARGET) == "H").mean().alias("P(H)"))
        .pivot(on=col, index="elo", values="P(H)")
        .select("elo", "low", "mid", "high")
        .sort("elo")
    )


for col in KEY[1:]:
    print(col)
    print(within_elo(eda, col))

The same question as a number: validate on the last training season, fit on the ones
before. Compare a multinomial logistic regression on `elo_diff` alone with one on
`elo_diff` plus each feature, on the same rows. A drop in log loss is what the feature
adds beyond Elo. This is a quick screen, not the model; feature_importance does that.

In [ ]:
VALID = eda["season_start"].max()


def fit_score(df: pl.DataFrame, cols: list[str]) -> float:
    fit, valid = (
        df.filter(pl.col("season_start") < VALID),
        df.filter(pl.col("season_start") == VALID),
    )
    scale = StandardScaler().fit(fit.select(cols).to_numpy())
    model = LogisticRegression(max_iter=1000).fit(
        scale.transform(fit.select(cols).to_numpy()), fit[TARGET].to_numpy()
    )
    p = model.predict_proba(scale.transform(valid.select(cols).to_numpy()))
    return log_loss(valid[TARGET].to_numpy(), p, labels=model.classes_)


rows = []
for col in KEY[1:]:
    rows_ok = eda.filter(pl.col(col).is_not_null() & pl.col("elo_diff").is_not_null())
    elo_only = fit_score(rows_ok, ["elo_diff"])
    both = fit_score(rows_ok, ["elo_diff", col])
    rows.append(
        {
            "feature": col,
            "games": rows_ok.height,
            "elo_only": elo_only,
            "elo_plus": both,
            "gain": elo_only - both,
        }
    )
added = pl.DataFrame(rows).sort("gain", descending=True)
added

## 5. Summary

One row per key feature: how often it is null, how far P(H) moves from its lowest to
its highest decile, and what it adds beyond Elo. Write the conclusions below and carry
them to feature_importance (issue #13): which features to keep, which to merge, and
which nulls deserve a flag column.

In [ ]:
summary = pl.DataFrame(
    [
        {
            "feature": col,
            "null_share": eda[col].null_count() / eda.height,
            "P(H) spread": (lambda d: d["H"].max() - d["H"].min())(by_decile(eda, col)),
        }
        for col in KEY
    ]
).join(added.select("feature", "gain"), on="feature", how="left")
summary

### Conclusions

- Baseline log loss (section 1):
- Nulls worth a flag (section 2):
- Features that move the result (section 3):
- Features that add nothing beyond Elo, or overlap another (section 4):